In [1]:
%pip install kagglehub opencv-python-headless numpy pandas tqdm scikit-learn matplotlib seaborn tensorflow -q



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponivel:", tf.config.list_physical_devices("GPU"))


In [ ]:
import kagglehub

path = kagglehub.dataset_download("tombackert/brain-tumor-mri-data")
dataset_path = Path(path)

INPUT_BASE = dataset_path / "brain-tumor-mri-dataset"
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
TARGET_SIZE = (224, 224)
BASE_DIR = Path.cwd()

## Fazendo a divisão em train/val/test (estratificado)

In [ ]:
def listar_arquivos_por_classe(input_base, classes):
    dados = {}
    for c in classes:
        class_dir = input_base / c
        arquivos = sorted([f for f in class_dir.glob("*") if f.is_file()])
        dados[c] = arquivos
    return dados


def criar_split(input_base, classes, val_size=0.15, test_size=0.15, seed=SEED):
    """Split estratificado train/val/test a partir das pastas de classe originais."""
    dados = listar_arquivos_por_classe(input_base, classes)
    registros = []

    for classe, arquivos in dados.items():
        if len(arquivos) == 0:
            raise ValueError(f"Nenhum arquivo encontrado para a classe '{classe}' em {input_base / classe}")

        train_val, test = train_test_split(
            arquivos, test_size=test_size, random_state=seed, shuffle=True
        )
        val_fraction_of_train_val = val_size / (1 - test_size)
        train, val = train_test_split(
            train_val, test_size=val_fraction_of_train_val, random_state=seed, shuffle=True
        )

        for f in train:
            registros.append({"filepath": str(f), "classe": classe, "split": "train"})
        for f in val:
            registros.append({"filepath": str(f), "classe": classe, "split": "val"})
        for f in test:
            registros.append({"filepath": str(f), "classe": classe, "split": "test"})

    return pd.DataFrame(registros)


df_split = criar_split(INPUT_BASE, CLASSES)
print(df_split.groupby(["split", "classe"]).size().unstack())
print("\nTotal de imagens:", len(df_split))

df_split.to_csv(BASE_DIR / "split_baseline.csv", index=False)


In [ ]:
def construir_cnn(input_shape=(224, 224, 1), n_classes=4, seed=SEED):
    tf.random.set_seed(seed)

    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(n_classes, activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


construir_cnn().summary()

In [ ]:
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
)
eval_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

def criar_generator(datagen, df, shuffle):
    return datagen.flow_from_dataframe(
        df,
        x_col="filepath",
        y_col="classe",
        target_size=TARGET_SIZE,
        color_mode="grayscale",
        classes=CLASSES,
        class_mode="categorical",
        batch_size=BATCH_SIZE,
        seed=SEED,
        shuffle=shuffle,
    )

train_gen = criar_generator(train_datagen, df_split[df_split["split"] == "train"], shuffle=True)
val_gen = criar_generator(eval_datagen, df_split[df_split["split"] == "val"], shuffle=False)
test_gen = criar_generator(eval_datagen, df_split[df_split["split"] == "test"], shuffle=False)


In [ ]:
EPOCHS = 40
RESULTADOS_DIR = BASE_DIR / "resultados_modelo_base"
RESULTADOS_DIR.mkdir(exist_ok=True)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASSES)),
    y=train_gen.classes,
)
class_weights = dict(enumerate(class_weights_array))
print("Class weights:", class_weights)

model = construir_cnn(input_shape=(*TARGET_SIZE, 1), n_classes=len(CLASSES))

checkpoint_path = RESULTADOS_DIR / "melhor_modelo.keras"
cbs = [
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    callbacks.ModelCheckpoint(str(checkpoint_path), monitor="val_loss", save_best_only=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=cbs,
    verbose=1,
)

pd.DataFrame(history.history).to_csv(RESULTADOS_DIR / "history.csv", index=False)


## Avaliação no conjunto de teste

In [ ]:
test_gen.reset()
y_true = test_gen.classes
y_pred_probs = model.predict(test_gen, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

report = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True)
print(classification_report(y_true, y_pred, target_names=CLASSES))

pd.DataFrame(report).transpose().to_csv(RESULTADOS_DIR / "classification_report.csv")
print(f"\nAcuracia no teste: {report['accuracy']:.4f}")


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de confusão — modelo base")
plt.tight_layout()
plt.savefig(RESULTADOS_DIR / "matriz_confusao.png", dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Acurácia por época")
axes[0].set_xlabel("Época")
axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss por época")
axes[1].set_xlabel("Época")
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTADOS_DIR / "curvas_treino.png", dpi=150)
plt.show()


## Próximos passos

Para comparar com seu pré-processamento: troque `INPUT_BASE` (Etapa 2) pelo caminho das suas imagens
já processadas — mantendo a mesma estrutura de pastas por classe — e rode o notebook novamente do
início. Como o split é estratificado e usa sempre `SEED = 42`, mas é gerado a partir da listagem de
arquivos de cada pasta, a comparação fica justa **desde que os nomes/quantidade de arquivos por classe
sejam os mesmos** entre o dataset original e o pré-processado (mesma imagem, só com conteúdo alterado).

Depois de rodar as duas vezes, compare diretamente os arquivos salvos em `resultados_modelo_base/`
(seria bom renomear a pasta entre as rodadas, ex. `resultados_original/` e `resultados_preprocessado/`,
para não sobrescrever) — principalmente `classification_report.csv` e a acurácia final no teste.
